# p113 — control & calibration for "weights move, representation holds"

This is the control run for the third p109 notebook ([weights move, representation holds](./p109_weights_move_representation_holds.ipynb)). p113/s999/ds598 is a clean grokker with **no ~27k event** — that destabilize-and-recover has only ever appeared in p109. So this notebook is not looking for the event; it asks which of p109's findings are **generic to grokking + weight decay** and which are **p109-specific**, and it begins a **calibration** baseline so later we can scan other variants for early/partial versions of the radial escape.

**p113 is the reference for "normal" grokking** — its onset lands inside the canonical on-time band (`early_grokking_epoch = 9000`, `late_grokking_epoch = 12000` in `views/cross_variant.py`), so it defines the normal window the others are judged against. p109, as we'll see, groks *early* — which turns out to matter.

Going in, the expectations were:

- **Should replicate (generic):** the plateau sits at **rotational equilibrium** (weights rotating at fixed norm). Kosson's universal consequence of weight decay, so it should be the most robust. Sharp version: since these variants share `lr = 1e-3, wd = 1.0`, p113's plateau **rotation rate** should be ≈ p109's — not just "both rotate," the *same* rate.
- **Should be absent (the control's negative):** no late radial escape, no late CKA dip — confirming the 27k escape is the event.
- **Genuinely open:** does p113 also pre-form its representation early and disorganize during grokking (the p109 pattern), or follow the textbook *memorize → reorganize at grokking* route?

p113's grokking marker is the cleanest available because its drop is sharp. We use the pipeline's definition — **`second_descent_onset`: the first epoch after the test-loss peak where the loss has fallen ≥ 80% of the way from peak toward zero** (`second_descent_onset_diff_threshold = 0.8`) — computed directly from the test-loss curve.

---
*Conventions follow the p109 notebooks; all access through the miscope API (`variant.test_losses`, `variant.run_with_cache`, `variant.artifacts`). The `=` query (position 2) is the readout site; CKA is gauge-invariant linear CKA on the full (a,b) grid. Comparisons to p109 are **grok-relative** (in units of each model's own onset), since p113 groks ~3× later.*

In [ ]:
import os
from pathlib import Path

import numpy as np
import plotly.graph_objects as go

root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data" / "modulo_addition_1layer").exists())
os.chdir(root)
from miscope.families.discovery import load_family_from_dir

fam = load_family_from_dir("data/modulo_addition_1layer", "data")
variant = fam.get_variant(prime=113, seed=999, data_seed=598)      # control / normal-timing reference
ref_p109 = fam.get_variant(prime=109, seed=485, data_seed=598)     # the (early) event variant, for contrast
NORMAL_BAND = (9000, 12000)   # canonical on-time grokking band (views/cross_variant.py)


def grok_onset(v):
    # second_descent_onset: first epoch after the test-loss peak where loss has dropped >=80% from peak.
    tl = np.asarray(v.test_losses); peak = float(tl.max()); pe = int(tl.argmax())
    hit = np.where((np.arange(len(tl)) >= pe) & ((peak - tl) / peak >= 0.8))[0]
    return (int(hit[0]) if len(hit) else None), peak, pe


def linear_cka(X, Y):
    # Gauge-invariant representational similarity in [0,1] (Kornblith et al. 2019).
    X = X - X.mean(0); Y = Y - Y.mean(0)
    return float(np.linalg.norm(Y.T @ X, "fro") ** 2 /
                 (np.linalg.norm(X.T @ X, "fro") * np.linalg.norm(Y.T @ Y, "fro")))


SITES = {"embed_a": ("embed.hook_out", 0), "attn_out": ("blocks.0.attn.hook_out", 2),
         "mlp_hidden": ("blocks.0.mlp.hook_out", 2), "resid_post": ("blocks.0.hook_out", 2)}
COLORS = {"embed_a": "#9467bd", "attn_out": "#d62728", "mlp_hidden": "#2ca02c", "resid_post": "#1f77b4"}


def site_reps(v, prime, epoch):
    grid = [[a, b] for a in range(prime) for b in range(prime)]
    _, cache = v.run_with_cache(v.make_probe(grid), epoch=int(epoch))
    return {n: cache[k][:, pos, :].detach().cpu().numpy().astype(np.float64) for n, (k, pos) in SITES.items()}


variant

## 1. Grokking onset — the pipeline marker

`second_descent_onset` = first epoch after the test-loss peak where the loss has fallen ≥ 80% from peak. (Test loss *spikes* during early memorization before it descends, so the marker measures the drop from that spike, not from initialization.)

In [ ]:
tl = np.asarray(variant.test_losses)
onset, peak, peak_ep = grok_onset(variant)

fig = go.Figure()
fig.add_vrect(x0=NORMAL_BAND[0], x1=NORMAL_BAND[1], fillcolor="green", opacity=0.07, line_width=0,
              annotation_text="normal-grokking band", annotation_position="top left")
fig.add_trace(go.Scatter(x=np.arange(len(tl)), y=tl, mode="lines", name="test loss", line=dict(color="#1f77b4")))
fig.add_vline(x=peak_ep, line=dict(color="gray", dash="dot"), annotation_text=f"peak {peak:.0f}")
fig.add_vline(x=onset, line=dict(color="#d62728", dash="dash"),
              annotation_text=f"onset {onset}", annotation_position="top right")
fig.update_layout(title="p113 test loss — onset = first epoch >=80% down from the memorization peak",
                  xaxis_title="epoch", yaxis_title="test loss", yaxis_type="log", height=420)
fig.show()
print(f"p113: test-loss peak {peak:.2f} at epoch {peak_ep}; second_descent_onset = epoch {onset} "
      f"(in normal band {NORMAL_BAND}: {NORMAL_BAND[0] <= onset <= NORMAL_BAND[1]})")
o109, pk109, _ = grok_onset(ref_p109)
print(f"p109 for contrast: onset = epoch {o109}  (early: before the normal band) -> p113 groks ~{onset/o109:.1f}x later")

## 2. Representation across training — pre-formed (p109) or formed at grokking (textbook)?

Gauge-invariant CKA-to-final at the four `=`-query sites. p109 (notebook 3) pre-formed its representation *during memorization* and merely locked it at grokking, with a brief disorganization dip. The fair test is **grok-relative** (p113 groks ~3× later), so after the per-site overview we compare both models in units of their own onset.

In [ ]:
all_ep = np.array(variant.artifacts.get_epochs("parameter_snapshot"))
want = [0, 500, 1500, 3000, 5000, 7000, 9000, 10000, 11000, 12000, 13000, 15000, 18000, 22000, int(all_ep[-1])]
cka_ep = sorted({int(all_ep[np.argmin(np.abs(all_ep - w))]) for w in want})
REPS = {e: site_reps(variant, 113, e) for e in cka_ep}
ref = cka_ep[-1]
cka = {s: np.array([linear_cka(REPS[e][s], REPS[ref][s]) for e in cka_ep]) for s in SITES}

fig = go.Figure()
fig.add_vrect(x0=NORMAL_BAND[0], x1=NORMAL_BAND[1], fillcolor="green", opacity=0.07, line_width=0)
fig.add_vline(x=onset, line=dict(color="gray", dash="dash"), annotation_text="onset")
for s in SITES:
    fig.add_trace(go.Scatter(x=cka_ep, y=cka[s], mode="lines+markers", name=s, line=dict(color=COLORS[s])))
fig.update_layout(title="p113 §2 — representation similarity to final (CKA); forms through the grokking band",
                  xaxis_title="epoch", yaxis_title="CKA to final", yaxis_range=[0, 1.02],
                  legend=dict(x=0.01, y=0.99), height=460)
fig.show()
ce = np.array(cka_ep)
for s in SITES:
    formed = ce[np.argmax(cka[s] >= 0.9)] if (cka[s] >= 0.9).any() else None
    print(f"{s:>11}: reaches CKA 0.90 by epoch {formed}")

In [ ]:
# Grok-relative contrast: resid_post CKA-to-final vs fraction of each model's own onset.
FRACS = [0.3, 0.5, 0.7, 0.9, 1.0, 1.2, 1.5, 2.0]
curves = {}
for tag, v, prime in [("p113 (normal)", variant, 113), ("p109 (early)", ref_p109, 109)]:
    o, _, _ = grok_onset(v)
    ep = np.array(v.artifacts.get_epochs("parameter_snapshot")); fin = int(ep[-1])
    Rf = site_reps(v, prime, fin)["resid_post"]
    ys = []
    for f in FRACS:
        e = int(ep[np.argmin(np.abs(ep - o * f))])
        ys.append(linear_cka(site_reps(v, prime, e)["resid_post"], Rf) if e <= fin else np.nan)
    curves[tag] = np.array(ys)

fig = go.Figure()
fig.add_vline(x=1.0, line=dict(color="gray", dash="dash"), annotation_text="grok onset")
for tag, c in zip(curves, ["#1f77b4", "#d62728"]):
    fig.add_trace(go.Scatter(x=FRACS, y=curves[tag], mode="lines+markers", name=tag, line=dict(color=c)))
fig.update_layout(title="resid_post CKA-to-final, grok-relative: p109 pre-forms (+dip), p113 forms at grokking",
                  xaxis_title="epoch / grok-onset", yaxis_title="resid_post CKA to final",
                  yaxis_range=[0, 1.02], legend=dict(x=0.01, y=0.99), height=440)
fig.show()
for tag in curves:
    c = curves[tag]; at_onset = c[FRACS.index(1.0)]
    dip = any(c[i] < c[i - 1] - 0.02 for i in range(1, len(c)))
    print(f"{tag}: CKA at onset = {at_onset:.2f} | pre-onset(0.5x) = {c[FRACS.index(0.5)]:.2f} | "
          f"non-monotonic dip near onset: {dip}")

## 3. Rotational equilibrium — and the calibration table

The plateau test and the headline prediction. Decompose each `W_in` step into **radial** (norm change) and **tangential** (rotation); equilibrium = stable norm, ongoing rotation, mostly tangential. Then place p113's clean post-lock plateau beside p109's pre-event plateau: same `lr`/`wd` should give the **same operating point**.

In [ ]:
def rotational(v):
    ps = np.array(v.artifacts.get_epochs("parameter_snapshot"))
    W = np.stack([v.artifacts.load_epoch("parameter_snapshot", int(e))["W_in"].T.astype(np.float64) for e in ps])
    norm = np.linalg.norm(W, axis=2)
    w0, w1 = W[:-1], W[1:]
    n0 = np.linalg.norm(w0, axis=2) + 1e-12; n1 = np.linalg.norm(w1, axis=2) + 1e-12
    ang = np.degrees(np.arccos(np.clip((w0 * w1).sum(2) / (n0 * n1), -1, 1)))
    dw = w1 - w0
    radial = (dw * (w0 / n0[:, :, None])).sum(2)
    tang = np.sqrt(np.clip((dw ** 2).sum(2) - radial ** 2, 0, None))
    return ps[:-1], norm, ang, radial, tang


mid, norm, ang, radial, tang = rotational(variant)
fig = go.Figure()
fig.add_vline(x=onset, line=dict(color="gray", dash="dash"), annotation_text="grok onset")
fig.add_trace(go.Scatter(x=mid, y=np.median(ang, 1), mode="lines", name="median angular update / step",
                         line=dict(color="#1f77b4")))
fig.add_trace(go.Scatter(x=mid, y=np.median(norm[:-1], 1), mode="lines", name="median ||W_in[j]||",
                         line=dict(color="#2ca02c"), yaxis="y2"))
fig.update_layout(title="p113 §3 — plateau rotational equilibrium (rotation continues at stable norm)",
                  xaxis_title="epoch", yaxis_title="angular update (deg/step)",
                  yaxis2=dict(title="median ||W_in[j]||", overlaying="y", side="right"),
                  legend=dict(x=0.99, y=0.99, xanchor="right"), height=440)
fig.show()

In [ ]:
# Calibration table (clean post-lock plateaus) + radial-escape check, both computed live.
def stats(v, lo, hi):
    mid, norm, ang, radial, tang = rotational(v)
    m = (mid >= lo) & (mid <= hi); post = mid >= lo
    tf = np.median(tang[m] / (np.sqrt(radial[m] ** 2 + tang[m] ** 2) + 1e-12))
    i = int(np.abs(radial[post]).max(1).argmax())
    return dict(ang=np.median(ang[m]), norm=np.median(norm[:-1][m]), rad=np.median(np.abs(radial[m])),
                tf=tf, maxrad=np.abs(radial[post]).max(), maxrad_ep=int(mid[post][i]))


s113 = stats(variant, 15000, 24000)     # p113 plateau, fully past the ~14k lock
s109 = stats(ref_p109, 8000, 26000)     # p109 pre-event plateau
print("ROTATIONAL EQUILIBRIUM CALIBRATION  (shared lr=1e-3, wd=1.0)\n")
print(f"{'':<18}{'angular(deg/step)':>18}{'norm':>8}{'|radial|':>10}{'tang.frac':>10}{'max|radial| after lock':>24}")
for nm, s in [("p113 (control)", s113), ("p109 (pre-event)", s109)]:
    print(f"{nm:<18}{s['ang']:>18.3f}{s['norm']:>8.3f}{s['rad']:>10.4f}{s['tf']:>10.2f}"
          f"{s['maxrad']:>16.3f} @ {s['maxrad_ep']}")
print(f"\nrotation-rate ratio p113/p109 = {s113['ang'] / s109['ang']:.2f}  (1.0 = identical operating point)")
print(f"radial escape: p113 max {s113['maxrad']:.3f} (= floor, no escape) vs p109 event 0.620 @ 27700 (nb3)")

## Reading — what replicated, what is p109-specific

The control did its job, and the corrected grok-relative comparison sharpens both conclusions.

**p113 is the normal-timing reference; p109 groks early.** p113's onset (≈9.5k) sits inside the canonical on-time band [9000, 12000]; p109's (≈3.6k) is well before it. p109 is an *early* grokker — and that is the thread that ties its other idiosyncrasies together.

**Replicated, and tightly calibrated — plateau rotational equilibrium is generic.** On its clean post-lock plateau p113's `W_in` rows rotate at **0.18°/step** with the radial part at the noise floor (~0.001) and ~78–83% of the motion tangential — rotational equilibrium, same as p109. The sharp prediction holds almost exactly: p113 vs p109 is **0.18 vs 0.19 °/step (ratio 0.95)**, norm 0.41 vs 0.42, radial floor 0.0010 vs 0.0009. The operating point is set by the shared `lr`/`wd`, not the task. "Weights move, the representation holds" is a **generic plateau property**, and the equilibrium is now calibrated across two variants.

**Absent — no radial escape, no late dip.** p113's largest post-lock radial step is **0.009** (the floor) versus p109's **0.620** at the event. No neuron breaks equilibrium; the representation holds flat to 25k. The 27k radial escape is confirmed as the **event**, p109-only.

**p109-specific (grok-relative, rigorous) — early pre-formation *and* the disorganization dip.** Measured in units of each model's own onset, the contrast is clean: p109 enters its transition with the representation already **~75% formed** (resid_post CKA 0.73–0.77 at 0.7–1.0× onset) and shows a **transient dip** (0.77 → 0.70 around 1.0–1.2× onset) before locking. p113 enters only **~34% formed** (0.05 → 0.10 at 0.3–0.5× onset) and rises **monotonically** through its transition to lock by ~1.5× onset — a dense scan shows *no* dip. So both p109 traits — the pre-formed representation and the during-grokking churn — are **idiosyncratic to p109**, not features of grokking. p113 is the textbook monotonic memorize→reorganize.

**What this re-frames.** The control splits notebook 3's claims cleanly. The **plateau** half ("weights rotate at equilibrium while the representation holds") is general and now calibrated to ~5%. The **formation** half ("representation forms early") is p109's, and it correlates with p109 being an *early* grokker: p109 has its generalizing representation substantially in place before it even groks, then churns briefly, then — much later — goes metastable at 27k. The open question is whether these are one story: *does early grokking come with early representation formation, and does that trajectory carry a higher chance of late metastability?*

**Calibration status & next steps:**

- **Equilibrium baseline is solid.** Two variants agree to ~5% on rotation rate and essentially exactly on norm / radial floor / tangential fraction. The **radial floor (~0.001)** is the scan quantity; p109's event is the loud tail. A shared threshold (rather than per-variant) looks viable, but one or two more clean grokkers would confirm it before a population scan.
- **Test the early-formation thread on another early grokker.** If a second early grokker also pre-forms its representation (high grok-relative CKA at onset), early-formation becomes a *regime* tied to grokking speed; if not, p109 is a case study. This is the cheap, decisive next probe.
- **`rotational_dynamics` can likely ship with a shared baseline**; `representation_similarity` should report grok-relative (onset-normalized) curves so cross-variant comparison is fair.